# 04 — Morphology, Stemming & Lemmatization

**Learning objective.** Understand how word forms relate and when aggressive normalization helps or hurts.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## Mental model

**word forms → stemming/lemmatization → canonical forms → reduced lexical variation**

The key question is not “which API do I call?” but **which representation changes next when I change a control?**

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Use **stemming** | forms merge aggressively | recall may improve but readable/semantic distinctions can collapse |
| Use **lemmatization** | forms merge using lexical/grammatical rules | cleaner canonical words but more linguistic machinery is needed |
| Preserve morphology | vocabulary grows | tense/number/form information remains available |

### Engineering rule
Change **one control at a time**, predict the direction of the effect, then measure whether reality matches the prediction.

> Before changing a parameter, state the expected direction of the downstream effect.

## Think before running the next cell

1. Should `better` map to `good` under stemming? Under lemmatization?
2. What happens to sentiment or tense information if you over-normalize forms?

Do not scroll to the output until you have an expected answer—even a rough one.

### When to use
Useful for sparse lexical systems where inflection creates unnecessary sparsity.

### When not to use / caution
Often unnecessary for contextual/subword models that can learn morphology directly.

### Debugging lens
Look for pairs the normalizer merged that the task actually needs to distinguish.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


**Stemming** applies heuristics to strip affixes and may produce non-words. **Lemmatization** aims to return a valid dictionary lemma using morphological/linguistic knowledge. Modern contextual models often need less aggressive normalization because morphology itself can be informative.

In [2]:
from nltk.stem import PorterStemmer, SnowballStemmer
words=['connect','connected','connecting','connection','studies','studying','better']
porter=PorterStemmer(); snow=SnowballStemmer('english')
pd.DataFrame({'word':words,
              'porter':[porter.stem(x) for x in words],
              'snowball':[snow.stem(x) for x in words]})

         word   porter snowball
0     connect  connect  connect
1   connected  connect  connect
2  connecting  connect  connect
3  connection  connect  connect
4     studies    studi    studi
5    studying    studi    studi
6      better   better   better

In [3]:
# Small rule-driven lemma demo; real lemmatizers use POS + lexical resources.
irregular={'better':'good','went':'go','children':'child'}
def educational_lemma(w):
    if w in irregular: return irregular[w]
    if w.endswith('ies'): return w[:-3]+'y'
    if w.endswith('ing') and len(w)>5: return w[:-3]
    return w
pd.DataFrame({'word':words+['went','children'],'educational_lemma':[educational_lemma(w) for w in words+['went','children']]})

         word educational_lemma
0     connect           connect
1   connected         connected
2  connecting           connect
3  connection        connection
4     studies             study
5    studying             study
6      better              good
7        went                go
8    children             child

---
## Production takeaways
- Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
- Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
- Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Contrast stemming and lemmatization
- Recognize when morphology should be preserved